# Ordenação

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx roda o Java com o seu JDK. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Acompanhar as intercalações do merge sort e explicar por que ele leva O(n log n)
- Escrever a etapa de intercalação e reaproveitá-la para contar inversões
- Ordenar por uma chave com um comparator e depois percorrer os dados ordenados numa passada
- Usar a ordenação por contagem quando os valores ficam num intervalo pequeno

## Em resumo

Ordenar deixa os dados fáceis de buscar, agrupar e comparar, e as principais ordenações O(n log n) usam uma de duas ideias.

O **merge sort** intercala sequências ordenadas. Intercalar duas listas ordenadas custa uma comparação por valor retirado, então uma passada por n valores custa uns n. Comece com sequências de 1 e dobre o tamanho a cada passada: depois de log₂ n passadas sobra uma sequência ordenada, com O(n log n) no total, sempre.

![As passadas do merge sort em 8 valores](passes.pt-br.svg)

**O(n²) contra O(n log n):** ordenação por seleção e `Arrays.sort` nos mesmos números aleatórios. Dobre `n` e compare.

In [ ]:
int n = 20_000;
int[] a = new Random(2).ints(n, 0, 1_000_000).toArray();
int[] b = a.clone();

long inicio = System.nanoTime();
for (int i = 0; i < n; i++) { // ordenação por seleção
    int min = i;
    for (int j = i + 1; j < n; j++) {
        if (a[j] < a[min]) {
            min = j;
        }
    }
    int t = a[i];
    a[i] = a[min];
    a[min] = t;
}
long meio = System.nanoTime();
Arrays.sort(b);
long fim = System.nanoTime();

IO.println("seleção:     " + (meio - inicio) / 1_000_000 + " ms");
IO.println("Arrays.sort: " + (fim - meio) / 1_000_000 + " ms");
IO.println("mesmo resultado: " + Arrays.equals(a, b));

O **quicksort** escolhe um pivô, move os valores menores para a esquerda dele e os maiores para a direita (uma **partição**), e ordena cada lado. Ele é O(n log n) em média, mas um pivô ruim toda vez, como o primeiro elemento de um array ordenado, o deixa O(n²).

**Um pivô ruim toda vez.** Este quicksort sempre escolhe o primeiro elemento. Conte as comparações em dados aleatórios e em dados já ordenados.

In [ ]:
// shared: quicksort com o primeiro elemento como pivô, contando comparações
long comparisons = 0;

void quicksort(int[] a, int lo, int hi) {
    if (lo >= hi) {
        return;
    }
    int pivo = a[lo];
    int i = lo;
    for (int j = lo + 1; j <= hi; j++) {
        comparisons++;
        if (a[j] < pivo) {
            i++;
            int t = a[i];
            a[i] = a[j];
            a[j] = t;
        }
    }
    int t = a[lo];
    a[lo] = a[i];
    a[i] = t;
    quicksort(a, lo, i - 1);
    quicksort(a, i + 1, hi);
}

In [ ]:
int n = 3000;
int[] aleatorio = new Random(3).ints(n, 0, 1_000_000).toArray();
quicksort(aleatorio, 0, n - 1);
IO.println(String.format("aleatório: %,d comparações", comparisons));

comparisons = 0;
int[] ordenado = new int[n];
for (int i = 0; i < n; i++) {
    ordenado[i] = i;
}
quicksort(ordenado, 0, n - 1);
IO.println(String.format("ordenado:  %,d comparações (n²/2 = %,d)", comparisons, (long) n * n / 2));

Quando os valores são inteiros pequenos, como notas de 0 a 100, a **ordenação por contagem** nem compara: ela conta quantas vezes cada valor aparece, em O(n + k) para k valores possíveis.

Em Java, `Arrays.sort` num `int[]` usa um quicksort, e em objetos um merge sort guiado por um **comparator**, como `Comparator.comparingInt(Meeting::start)`. Essa ordenação é **estável**: elementos iguais mantêm a ordem em que estavam.

Cuidado: um comparator escrito como `a - b` pode estourar. Use `Integer.compare(a, b)`.

**Comparators:** pelo tamanho, depois em ordem alfabética, e invertido. A última linha mostra o estouro de `a - b`.

In [ ]:
List<String> palavras = new ArrayList<>(List.of("pear", "fig", "apple", "kiwi", "date"));
palavras.sort(Comparator.comparingInt(String::length).thenComparing(Comparator.naturalOrder()));
IO.println(palavras);
palavras.sort(Comparator.comparingInt(String::length).reversed());
IO.println(palavras);

List<Integer> numeros = new ArrayList<>(List.of(2_000_000_000, -1_000_000_000, 0));
numeros.sort((a, b) -> a - b); // estoura
IO.println("com a - b:            " + numeros);
numeros.sort(Integer::compare);
IO.println("com Integer.compare:  " + numeros);

## Leituras

- [Sorting](https://visualgo.net/en/sorting) (VisuAlgo (National University of Singapore), 20 min): Escolha Merge Sort, Quick Sort e Counting Sort no menu de cima e rode cada um num array curto seu. Repare na intercalação pegando o menor valor da frente, na partição movendo valores em volta do pivô e na ordenação por contagem nunca comparando dois valores.
- [Writing and Combining Comparators](https://dev.java/learn/language/fp/lambdas/writing-comparators/) (dev.java (Oracle), 8 min): Comparator.comparing e comparingInt montam um comparator a partir de uma chave, thenComparing desempata com uma segunda chave e reversed() inverte a ordem. Repare por que (i1 - i2) é um comparator ruim.

## Verifique

1. Quantas passadas o merge sort de baixo para cima faz em 1.000 valores?
2. Por que a ordenação por contagem não ordena nomes?
3. O que uma ordenação estável mantém, e quando isso importa?

## Desafios

Resolva os desafios da unidade aqui mesmo: cada célula começa com `// sphinx: challenge …`, e rodá-la roda **todos** os testes do desafio, inclusive os ocultos. Passar conta no seu progresso na barra lateral do Sphinx, como o Submit. A descrição completa, com exemplos e dicas, abre pela barra lateral.

### Notas da prova · Fácil

In [ ]:
// sphinx: challenge exam-scores
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] parts = IO.readln().trim().split(" +");
    int[] scores = new int[n];
    for (int i = 0; i < n; i++) {
        scores[i] = Integer.parseInt(parts[i]);
    }

    // TODO: conte cada nota, imprima as contagens de 100 até 0 e depois a mediana
}

### Rastro do merge · Médio

In [ ]:
// sphinx: challenge merge-trace
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] parts = IO.readln().trim().split(" +");
    int[] a = new int[n];
    for (int i = 0; i < n; i++) {
        a[i] = Integer.parseInt(parts[i]);
    }

    // TODO: intercale sequências de largura 1, 2, 4, ..., imprima o array depois de cada passada e depois as comparações
}

### Agenda unificada · Médio

In [ ]:
// sphinx: challenge calendar-merge
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    int[][] meetings = new int[n][2];
    for (int i = 0; i < n; i++) {
        String[] parts = IO.readln().trim().split(" +");
        meetings[i][0] = Integer.parseInt(parts[0]);
        meetings[i][1] = meetings[i][0] + Integer.parseInt(parts[1]); // the end
    }

    // TODO: ordene as reuniões pelo início, junte as que se sobrepõem em blocos e imprima as três linhas
}

### Distância entre rankings · Difícil

In [ ]:
// sphinx: challenge ranking-distance
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] parts = IO.readln().trim().split(" +");
    int[] ranking = new int[n];
    for (int i = 0; i < n; i++) {
        ranking[i] = Integer.parseInt(parts[i]);
    }

    // TODO: conte as inversões com um merge sort e depois imprima-as e a concordância
}